In [1]:
from typing import Any

import torch
import torch.nn as nn
import pandas as pd
from torch.utils.data import DataLoader , TensorDataset
import numpy as np
from sklearn.preprocessing import StandardScaler

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [3]:
btc_1d = pd.read_csv("../data/btc/btc_1d_data_2018_to_2025.csv")
btc_4h = pd.read_csv("../data/btc/btc_4h_data_2018_to_2025.csv")
btc_1h = pd.read_csv("../data/btc/btc_1h_data_2018_to_2025.csv")
btc_15m = pd.read_csv("../data/btc/btc_15m_data_2018_to_2025.csv")

In [4]:
btc_1d.head()

,Open time,Open,High,Low,Close,Volume,Close time,Quote asset volume,Number of trades,Taker buy base asset volume,Taker buy quote asset volume,Ignore
0,2018-01-01 00:00:00.000000 UTC,13715.65,13818.55,12750.00,13380.00,8609.915844,2018-01-01 23:59:59.999000 UTC,1.147997e+08,105595,3961.938946,5.280975e+07,0
1,2018-01-02 00:00:00.000000 UTC,13382.16,15473.49,12890.02,14675.11,20078.092111,2018-01-02 23:59:59.999000 UTC,2.797171e+08,177728,11346.326739,1.580801e+08,0
2,2018-01-03 00:00:00.000000 UTC,14690.00,15307.56,14150.00,14919.51,15905.667639,2018-01-03 23:59:59.999000 UTC,2.361169e+08,162787,8994.953566,1.335873e+08,0
3,2018-01-04 00:00:00.000000 UTC,14919.51,15280.00,13918.04,15059.54,21329.649574,2018-01-04 23:59:59.999000 UTC,3.127816e+08,170310,12680.812951,1.861168e+08,0
4,2018-01-05 00:00:00.000000 UTC,15059.56,17176.24,14600.00,16960.39,23251.491125,2018-01-05 23:59:59.999000 UTC,3.693220e+08,192969,13346.622293,2.118299e+08,0


In [5]:
feature_cols = ["Open", "High", "Low", "Close", "Volume"]

In [6]:
btc_1d = btc_1d[feature_cols]
btc_1d["Target_Ret"] = btc_1d["Close"].pct_change()

In [7]:
# train_features_scaled = feature_scaler.fit_transform()

In [8]:
btc_1d = btc_1d.dropna().reset_index(drop=True)

In [9]:
btc_1d.head()

,Open,High,Low,Close,Volume,Target_Ret
0,13382.16,15473.49,12890.02,14675.11,20078.092111,0.096794
1,14690.00,15307.56,14150.00,14919.51,15905.667639,0.016654
2,14919.51,15280.00,13918.04,15059.54,21329.649574,0.009386
3,15059.56,17176.24,14600.00,16960.39,23251.491125,0.126222
4,16960.39,17143.13,16011.21,17069.79,18571.457508,0.006450


In [10]:
btc_1d.dtypes

Open          float64
High          float64
Low           float64
Close         float64
Volume        float64
Target_Ret    float64
dtype: object

In [11]:
# btc_1d["Open time"] = pd.to_datetime(btc_1d["Open time"],utc=True)

In [12]:
# btc_1d.head()

In [13]:
# btc_1d.dtypes

In [14]:
btc_1d.shape

(3088, 6)

In [15]:
split_idx = int(0.8 * btc_1d.shape[0])
feature_scaler = StandardScaler()

train_set = feature_scaler.fit_transform(btc_1d[:split_idx])
test_set = feature_scaler.transform(btc_1d[split_idx:])

In [16]:
train_set.shape

(2470, 6)

In [17]:
test_set.shape

(618, 6)

In [18]:
seq_len = 21
x_train , y_train = [] , []
for i in range(train_set.shape[0] - seq_len):
   window = train_set[i : i + seq_len]
   target = train_set[i+seq_len,-1]
   x_train.append(window)
   y_train.append(target)

x_test , y_test = [] , []
for i in range(test_set.shape[0] - seq_len):
   window = test_set[i : i + seq_len]
   target = test_set[i+seq_len,-1]
   x_test.append(window)
   y_test.append(target)



In [19]:
y_test[0]

np.float64(0.7543224061258241)

In [20]:
x_train[0].shape

(21, 6)

In [21]:
len(x_train)

2449

In [22]:
len(y_train)

2449

In [23]:
y_train[0]

np.float64(0.06499418853042918)

In [24]:
x_arr = np.array(x_train,dtype=np.float32)
y_arr = np.array(y_train,dtype=np.float32).reshape(-1,1)

x_arr_test = np.array(x_test,dtype=np.float32)
y_arr_test = np.array(y_test,dtype=np.float32).reshape(-1,1)

print(len(x_arr_test) , len(y_arr_test))

597 597


In [25]:
print(len(x_arr) , len(y_arr))

2449 2449


In [26]:
train_dataset = TensorDataset(
    torch.from_numpy(x_arr[:split_idx]),
    torch.from_numpy(y_arr[:split_idx])
    )

test_dataset = TensorDataset(
    torch.from_numpy(x_arr_test),
    torch.from_numpy(y_arr_test)
    )

train_loader = DataLoader(train_dataset,batch_size= 32,shuffle=True)
test_loader = DataLoader(test_dataset,batch_size= 32,shuffle=False)

In [27]:
class VanillaRnn(nn.Module):
    def __init__(self,input_size ,hidden_size,num_layers):
        super().__init__()
        self.rnn = nn.LSTM(
            input_size= input_size,
            hidden_size = hidden_size,
            num_layers= num_layers,
            batch_first= True
        )

        self.fc = nn.Linear(hidden_size , 1)

    def forward(self,x):
        out , _ = self.rnn(x)
        last_step = out[:, -1, :]
        predict = self.fc(last_step)
        return predict

In [28]:
# import gc, torch
# gc.collect()
# torch.cuda.empty_cache()

In [29]:
# print(torch.cuda.mem_get_info())

In [30]:
model = VanillaRnn(6,20,2).to(device)

In [31]:
criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(),lr = 0.001, weight_decay=1e-4)

In [32]:
epochs = 150
correct_loss = float("inf")
for epoch in range(epochs):
    model.train()
    train_loss = 0
    for batch_x , batch_y in train_loader:
        batch_x , batch_y = batch_x.to(device) , batch_y.to(device)

        optimizer.zero_grad()
        out = model(batch_x)
        loss = criterion(out,batch_y)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * len(batch_x)

    train_loss /= len(train_loader.dataset)
    if correct_loss > train_loss:
        torch.save(model.state_dict(),"best_lstm.pt")
        print(f"save!!! , Epoch {epoch:02d}/{epochs} | Train MSE Loss: {train_loss:.6f}")
        correct_loss = train_loss


    if epoch % 10 == 0 or epoch == 1 or epoch == epochs-1:
        print(f"Epoch {epoch:02d}/{epochs} | Train MSE Loss: {train_loss:.6f}")




save!!! , Epoch 00/150 | Train MSE Loss: 0.970948
Epoch 00/150 | Train MSE Loss: 0.970948
save!!! , Epoch 01/150 | Train MSE Loss: 0.970709
Epoch 01/150 | Train MSE Loss: 0.970709
save!!! , Epoch 02/150 | Train MSE Loss: 0.970259
save!!! , Epoch 04/150 | Train MSE Loss: 0.969885
save!!! , Epoch 06/150 | Train MSE Loss: 0.969536
save!!! , Epoch 08/150 | Train MSE Loss: 0.969094
save!!! , Epoch 10/150 | Train MSE Loss: 0.968971
Epoch 10/150 | Train MSE Loss: 0.968971
save!!! , Epoch 12/150 | Train MSE Loss: 0.967927
save!!! , Epoch 13/150 | Train MSE Loss: 0.967869
save!!! , Epoch 14/150 | Train MSE Loss: 0.967290
save!!! , Epoch 15/150 | Train MSE Loss: 0.966702
save!!! , Epoch 16/150 | Train MSE Loss: 0.963620
save!!! , Epoch 18/150 | Train MSE Loss: 0.962775
save!!! , Epoch 19/150 | Train MSE Loss: 0.961131
save!!! , Epoch 20/150 | Train MSE Loss: 0.960676
Epoch 20/150 | Train MSE Loss: 0.960676
save!!! , Epoch 21/150 | Train MSE Loss: 0.959753
save!!! , Epoch 22/150 | Train MSE Loss:

In [39]:
ret_idx = 5  # ستون Target_Ret
r_mean = feature_scaler.mean_[ret_idx]
r_scale = feature_scaler.scale_[ret_idx]

model.load_state_dict(torch.load("best_lstm.pt", map_location=device))
model.eval()
preds_scaled, actuals_scaled = [], []

with torch.no_grad():
    for batch_x, batch_y in test_loader:
        batch_x = batch_x.to(device)
        out = model(batch_x).cpu().numpy()
        preds_scaled.append(out)
        actuals_scaled.append(batch_y.numpy())

preds_scaled = np.vstack(preds_scaled)
actuals_scaled = np.vstack(actuals_scaled)

# برگرداندن به بازده واقعی
pred_ret = preds_scaled * r_scale + r_mean
act_ret = actuals_scaled * r_scale + r_mean

mae = np.abs(pred_ret - act_ret).mean()
rmse = np.sqrt(np.mean((pred_ret - act_ret) ** 2))
dir_acc = (np.sign(pred_ret) == np.sign(act_ret)).mean() * 100

# baseline ها برای مقایسه
mae_zero = np.abs(act_ret).mean()           # پیش‌بینی بازده صفر
mae_mean = np.abs(act_ret - r_mean).mean()  # پیش‌بینی میانگین train
dir_up = (act_ret > 0).mean() * 100         # همیشه "بالا"

print(f"MAE بازده: {mae:.5f} | baseline صفر: {mae_zero:.5f} | baseline میانگین: {mae_mean:.5f}")
print(f"RMSE بازده: {rmse:.5f}")
print(f"دقت جهت: {dir_acc:.2f}% | baseline همیشه بالا: {dir_up:.2f}%")

# نمایش ۵ نمونه اول (به درصد)
for i in range(min(5, len(pred_ret))):
    real_val = act_ret[i, 0] * 100
    pred_val = pred_ret[i, 0] * 100
    diff_val = pred_val - real_val
    print(f"نمونه {i}: واقعی = {real_val:+.2f}% | پیش‌بینی = {pred_val:+.2f}% | اختلاف = {diff_val:+.2f}%")

MAE بازده: 0.02194 | baseline صفر: 0.01655 | baseline میانگین: 0.01660
RMSE بازده: 0.02858
دقت جهت: 50.42% | baseline همیشه بالا: 49.41%
نمونه 0: واقعی = +2.85% | پیش‌بینی = -0.45% | اختلاف = -3.30%
نمونه 1: واقعی = +3.97% | پیش‌بینی = -0.46% | اختلاف = -4.43%
نمونه 2: واقعی = -0.54% | پیش‌بینی = -1.18% | اختلاف = -0.64%
نمونه 3: واقعی = -2.84% | پیش‌بینی = -1.28% | اختلاف = +1.56%
نمونه 4: واقعی = -1.13% | پیش‌بینی = -1.39% | اختلاف = -0.25%


In [41]:
ret_idx = 5  # ستون Target_Ret
r_mean = feature_scaler.mean_[ret_idx]
r_scale = feature_scaler.scale_[ret_idx]

model.eval()
preds_scaled, actuals_scaled = [], []

with torch.no_grad():
    for batch_x, batch_y in test_loader:
        batch_x = batch_x.to(device)
        out = model(batch_x).cpu().numpy()
        preds_scaled.append(out)
        actuals_scaled.append(batch_y.numpy())

preds_scaled = np.vstack(preds_scaled)
actuals_scaled = np.vstack(actuals_scaled)

# بازده واقعی و پیش‌بینی‌شده (بدون مقیاس قیمت)
pred_ret = preds_scaled * r_scale + r_mean
act_ret = actuals_scaled * r_scale + r_mean

# دقت جهت: علامت بازده پیش‌بینی‌شده با علامت بازده واقعی یکی است؟
dir_acc = (np.sign(pred_ret) == np.sign(act_ret)).mean() * 100

# baseline: همیشه "بالا"
dir_up = (act_ret > 0).mean() * 100

print(f"Directional Accuracy: {dir_acc:.2f}%")
print(f"بیس لاین همیشه بالا: {dir_up:.2f}%")

Directional Accuracy: 50.42%
بیس لاین همیشه بالا: 49.41%
